# Guide 9 — The Robot Arm (Genesis)

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

Some arenas have "Genesis," a **pretend robot arm** on a screen that waves around
when you flip cards. It looks cool — but it does **not** affect who wins. It's
decoration only.

Because it's just for show, every one of these functions is written to *never*
crash the real game. If the robot arm has a problem, the code quietly ignores it and
keeps playing. That "quietly ignore errors" pattern is the main lesson here.

The code below is real code from the match program. It's part of the bigger match
"client," so it uses `self` — that just means it belongs to the main program object.


### How this guide fits in

**Depends on:** Guide 1. Optional and decoration-only — nothing else needs it.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Connecting, waving, and disconnecting

`_connect_genesis` joins the robot scene. `_genesis_flip_card` makes the arm move
when you flip a card. `_genesis_end_turn` and `_disconnect_genesis` finish up. Notice
that every function is wrapped in `try/except` so a robot problem never hurts your
match.


In [ ]:
    def _connect_genesis(self):
        """Best-effort: joins Genesis's competition-mode scene as our
        assigned team so _genesis_flip_card/_genesis_end_turn below can
        animate our arm. Any failure (pynqsim not installed, server
        unreachable, already joined) is logged and swallowed, never
        raised -- Genesis is purely cosmetic and never gets a vote in the
        real match, which is decided entirely by report_result."""
        self.genesis_sim = None
        if not (self.genesis_team_id and self.genesis_url):
            return
        try:
            from pynqsim import SimulationClient
            from urllib.parse import urlparse
            parsed = urlparse(self.genesis_url)
            sim = SimulationClient(parsed.hostname, port=parsed.port or 9002)
            sim.join_competition(team_id=self.genesis_team_id)
            self.genesis_sim = sim
            print(f'[genesis] joined as {self.genesis_team_id} at {self.genesis_url}')
        except Exception as exc:
            print(f'[genesis] connection skipped (cosmetic only, match unaffected): {exc}')
            self.genesis_sim = None

    def _genesis_flip_card(self, pos):
        """Mirrors a real flip onto the simulated arm, purely for visual
        effect -- the referee's own report_result claim is what actually
        decides the match either way."""
        if self.genesis_sim is None:
            return
        try:
            row, col = parse_pos(pos)
            self.genesis_sim.flip_card(row, col)
        except Exception as exc:
            print(f'[genesis] flip_card({pos}) failed (cosmetic only): {exc}')

    def _genesis_end_turn(self):
        if self.genesis_sim is None:
            return
        try:
            self.genesis_sim.end_turn()
        except Exception as exc:
            print(f'[genesis] end_turn failed (cosmetic only): {exc}')

    def _disconnect_genesis(self):
        if self.genesis_sim is None:
            return
        try:
            self.genesis_sim.leave_competition()
        except Exception as exc:
            print(f'[genesis] leave_competition failed (cosmetic only): {exc}')
        self.genesis_sim = None

### Check yourself

1. Why does every robot-arm function hide its errors instead of stopping the game?
2. What actually decides who wins a match, if not the robot arm?
